# DISTANCIA LOGICA HERVASIANA TESTEADA


## 1. Deconstrucción
Tu solicitud busca la validación empírica de una corrección geométrica. La premisa es que para "entender" la matemática, debemos ver la colisión entre el **Modelo Plano** (Euclidiano) y el **Modelo Esférico** (Haversine). 

* **El fallo oculto del modelo plano:** Estás asumiendo que un grado de longitud en el Ecuador mide lo mismo que un grado de longitud en Noruega. En un plano, los meridianos son paralelos; en la realidad, convergen en los polos. Al usar Pitágoras puro, estás tratando a la Tierra como un mapa de *Mercator*, lo que infla las distancias artificialmente a medida que te alejas del Ecuador.

---

## 2. Indagación Escéptica
* Si una simulación de red arroja un error del **24%** en la distancia, ¿qué valor tiene el resto de tu algoritmo de optimización? 
* ¿Puede un modelo de tráfico ser considerado "inteligente" si su base de cálculo de retardo ($delay$) está físicamente falseada por casi 500 kilómetros en un solo enlace?
* ¿Estamos optimizando una red real o una "red de papel" que solo existe en un plano bidimensional inexistente?

---

## 3. Contra-Perspectiva
Alguien podría argumentar que para redes metropolitanas (LAN o MAN) de menos de $10 \text{ km}$, la diferencia entre Haversine y Pitágoras es despreciable. En ese caso, la complejidad de Haversine es un desperdicio de CPU. Sin embargo, como tu proyecto usa **SNDLib** (que contiene backbones nacionales como *Germany50* o continentales como *Nobel-EU*), la perspectiva plana es académicamente indefendible.

---

## 4. Sinfonía Matemática: El Ejercicio Real

Vamos a conectar dos nodos reales de una topología de red típica: **Madrid** y **Berlín**.

| Ciudad | Longitud ($\lambda$) | Latitud ($\phi$) |
| :--- | :--- | :--- |
| **Madrid** | $-3.7038^\circ$ | $40.4168^\circ$ |
| **Berlín** | $13.4050^\circ$ | $52.5200^\circ$ |



### A. El Camino Errado (Pitágoras / Plano)
1.  **Diferencia de Grados:**
    * $\Delta \lambda (\text{lon}) = 13.4050 - (-3.7038) = 17.1088^\circ$
    * $\Delta \phi (\text{lat}) = 52.5200 - 40.4168 = 12.1032^\circ$
2.  **Hipotenusa en grados:**
    * $dist_{deg} = \sqrt{17.1088^2 + 12.1032^2} \approx 20.957^\circ$
3.  **Conversión a km ($1^\circ \approx 111 \text{ km}$):**
    * $Distancia = 20.957 \times 111 = \mathbf{2326.23 \text{ km}}$

### B. El Camino Real (Haversine / Esférico)
Aplicando tu nueva función (que considera que la Tierra se curva y los meridianos se estrechan):
1.  **Cálculo:** Se ajusta la longitud por el coseno de la latitud para compensar la convergencia de meridianos.
2.  **Resultado Geodésico:**
    * $Distancia = \mathbf{1869.15 \text{ km}}$

---

## Desarrollo Matemático Paso a Paso: Madrid -> Berlín

Para entender por qué la fórmula de **Haversine** es superior, vamos a desglosar el cálculo real entre **Madrid (M)** y **Berlín (B)**. Aquí verás cómo la trigonometría esférica ajusta la distancia que el cálculo plano infla.

### 1. Datos de Entrada (Coordenadas)
* **Madrid ($P_1$):** Latitud $\phi_1 = 40.4168^\circ$, Longitud $\lambda_1 = -3.7038^\circ$
* **Berlín ($P_2$):** Latitud $\phi_2 = 52.5200^\circ$, Longitud $\lambda_2 = 13.4050^\circ$
* **Radio de la Tierra ($R$):** $6371.0 \text{ km}$

### 2. Conversión a Radianes
La computadora no entiende grados para senos y cosenos. Multiplicamos por $\frac{\pi}{180}$.
* $\phi_1 = 0.7054 \text{ rad}$
* $\phi_2 = 0.9166 \text{ rad}$
* $\Delta\phi (\text{Lat}) = 0.9166 - 0.7054 = 0.2112 \text{ rad}$
* $\Delta\lambda (\text{Lon}) = 0.2339 - (-0.0646) = 0.2985 \text{ rad}$

---

### 3. El Corazón del Cálculo: La variable $a$
Aquí es donde ocurre la "magia". La fórmula de Haversine calcula el cuadrado de la mitad de la distancia de la cuerda:

$$a = \sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1) \cdot \cos(\phi_2) \cdot \sin^2\left(\frac{\Delta\lambda}{2}\right)$$

**Sustituyendo valores:**
1.  $\sin^2(0.2112 / 2) = \sin^2(0.1056) \approx \mathbf{0.0111}$
2.  $\cos(0.7054) \cdot \cos(0.9166) = 0.7614 \cdot 0.6084 \approx \mathbf{0.4633}$
3.  $\sin^2(0.2985 / 2) = \sin^2(0.1492) \approx \mathbf{0.0221}$

**Resultado de $a$:**
$$a = 0.0111 + (0.4633 \cdot 0.0221) = 0.0111 + 0.0102 = \mathbf{0.0213}$$

> **Nota Crítica:** En el cálculo plano (Pitágoras), el término $\cos(\phi_1) \cdot \cos(\phi_2)$ no existe. Ese término es el que **"encoge"** la importancia de la longitud a medida que subes al norte. En Berlín, los grados de longitud están más cerca entre sí que en Madrid.

---

### 4. Cálculo de la Distancia Angular ($c$)
Usamos la función `atan2` para hallar el ángulo central en radianes:

$$c = 2 \cdot \operatorname{atan2}(\sqrt{a}, \sqrt{1-a})$$
$$c = 2 \cdot \operatorname{atan2}(\sqrt{0.0213}, \sqrt{0.9787})$$
$$c = 2 \cdot \operatorname{atan2}(0.1459, 0.9893) \approx \mathbf{0.2934 \text{ rad}}$$



---

### 5. Resultado Final
Multiplicamos el ángulo por el radio terrestre:

$$Distancia = R \cdot c = 6371.0 \cdot 0.2934 = \mathbf{1869.15 \text{ km}}$$

---

## Resumen de la Diferencia

| Método | Lógica | Resultado | ¿Por qué falla/acierta? |
| :--- | :--- | :--- | :--- |
| **Pitágoras (Tu anterior)** | Plano (Papel) | $2326.23 \text{ km}$ | Ignora que la Tierra se curva y "estira" la longitud. |
| **Haversine (Tu nueva)** | Esférico (Real) | $1869.15 \text{ km}$ | **Corrige el ancho de los meridianos** según la latitud. |

**Veredicto:** Al usar Haversine en tu extractor, tu algoritmo "ve" que la red es un **24% más corta** de lo que pensabas. Esto significa que tu delay calculado será menor, lo que permite meter más tráfico en la fibra antes de violar los límites de QoS de tu paper.

¿Quieres que apliquemos ahora este cálculo de distancia para determinar el **Propagation Delay** ($\tau$) exacto en microsegundos para un enlace de fibra óptica?


## 5. Síntesis: Sanity Check Final

| Métrica | Resultado |
| :--- | :--- |
| **Error Absoluto** | $457.08 \text{ km}$ |
| **Error Relativo** | **$24.45\%$** |
| **Delay en Fibra (Errado)** | $11.63 \text{ ms}$ |
| **Delay en Fibra (Real)** | $9.34 \text{ ms}$ |

**Conclusión:**
El "sentido matemático" es que en el cálculo errado, estás sumando manzanas (grados de latitud) con naranjas (grados de longitud que se encogen). Haversine normaliza ambos ejes sobre la superficie de una esfera. Sin esta corrección, tu algoritmo penalizaría injustamente los enlaces que viajan de Este a Oeste en latitudes altas, tomando decisiones de enrutamiento basadas en una ilusión geográfica.

¿Deseas que procedamos ahora a integrar este cálculo en la función de **enrutamiento de demandas** para ver cómo cambia la carga de la red?

```

### FORMULA MATEMATICA HERVASINE ORIGINAL 

### La Matemática de $c$: El Ángulo Central

La variable $c$ representa la **distancia angular** entre los dos puntos, expresada en radianes. Matemáticamente, se deriva de la función arcoseno, pero se usa `atan2` por estabilidad numérica.

**La fórmula desglosada:**

1.  **$\sqrt{a}$**: Es el seno de la mitad del ángulo central ($\sin(c/2)$).
2.  **$\sqrt{1-a}$**: Es el coseno de la mitad del ángulo central ($\cos(c/2)$).
3.  **$\operatorname{atan2}(y, x)$**: Calcula el arco tangente de $y/x$, devolviendo el ángulo exacto en el cuadrante correcto.

**Relación Trigonométrica:**
Como $\tan(\theta) = \frac{\sin(\theta)}{\cos(\theta)}$, entonces:
$$\frac{c}{2} = \arctan\left(\frac{\sqrt{a}}{\sqrt{1-a}}\right)$$

Al multiplicar por **2**, recuperas el ángulo total $c$ que, al multiplicarse por el radio $R$, te da la longitud del arco (distancia real).



¿Deseas que integremos esta lógica ahora en tu función de cálculo de **Delay** para fibra óptica?

### Fórmula de Haversine: Estructura Matemática

Esta es la representación formal de la lógica que has integrado en tu extractor de topología:

$$a = \sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1) \cdot \cos(\phi_2) \cdot \sin^2\left(\frac{\Delta\lambda}{2}\right)$$

$$c = 2 \cdot \operatorname{atan2}\left(\sqrt{a}, \sqrt{1-a}\right)$$

$$d = R \cdot c$$



**Definición de Variables:**

* **$\phi_1, \phi_2$**: Latitud de los puntos 1 y 2 (convertidos obligatoriamente a **radianes**).
* **$\Delta\phi$**: Diferencia de latitudes ($\phi_2 - \phi_1$).
* **$\Delta\lambda$**: Diferencia de longitudes ($\lambda_2 - \lambda_1$).
* **$a$**: El cuadrado de la mitad de la distancia de la cuerda entre los puntos (valor entre 0 y 1).
* **$c$**: Distancia angular en radianes (el ángulo central).
* **$R$**: Radio medio de la Tierra ($6371$ km).
* **$d$**: Distancia final sobre la superficie del esferoide.

**¿Deseas que pasemos ahora a la función que utiliza esta $d$ para calcular el retardo de propagación ($\tau = d / v$) en tu algoritmo de tráfico?**